In [ ]:
#r "nuget: Azure.AI.Projects, 1.2.0-beta.5"
#r "nuget: Azure.Identity, 1.18.0-beta.2"
#r "nuget: Microsoft.Agents.AI, 1.0.0-rc1"
#r "nuget: Microsoft.Agents.AI.AzureAI, 1.0.0-rc1"
#r "nuget: ModelContextProtocol, 0.4.1-preview.1"

using Azure.AI.Projects;
using Azure.Identity;
using Microsoft.Agents.AI;
using System;
using System.Threading;
using System.Threading.Tasks;
using System.Collections.Generic;
using ModelContextProtocol.Client;
using Microsoft.Extensions.AI;

var endpointUrl = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_ENDPOINT");
var endpoint = endpointUrl != null ? new Uri(endpointUrl) : throw new InvalidOperationException("AZURE_FOUNDRY_PROJECT_ENDPOINT environment variable is not set.");
var deploymentName = "gpt-4o";
var apiKey = Environment.GetEnvironmentVariable("AZURE_FOUNDRY_PROJECT_API_KEY") ?? throw new InvalidOperationException("AZURE_FOUNDRY_PROJECT_API_KEY environment variable is not set.");
var githubToken = Environment.GetEnvironmentVariable("GITHUBTOKEN") ?? throw new InvalidOperationException("GITHUB_TOKEN environment variable is not set.");

const string agentName = "AgentFrameworkAgent-1";

var credOptions = new DefaultAzureCredentialOptions
{
TenantId = "16b3c013-d300-468d-ac64-7eda0820b6d3" // Specify the desired tenant ID
};

// Get a client to create/retrieve/delete server side agents with Azure Foundry Agents.
AIProjectClient aiProjectClient = new AIProjectClient(new Uri(endpointUrl), new DefaultAzureCredential(credOptions));

McpClient mcpClient = await McpClient.CreateAsync(new HttpClientTransport(new HttpClientTransportOptions
{
   TransportMode = HttpTransportMode.StreamableHttp,
   Endpoint = new Uri("https://api.githubcopilot.com/mcp"),
   AdditionalHeaders = new Dictionary<string, string>()
     {
         {  "Authorization", githubToken }
     }
}));

IList<McpClientTool> tools = await mcpClient.ListToolsAsync();

Console.WriteLine($"Tools Count: {tools.Count}");
 

AIAgent mcpAgent = await aiProjectClient.CreateAIAgentAsync(name: agentName, model: deploymentName, instructions: "You are a github expert.", tools: tools.Cast<AITool>().ToList());

AgentSession mcpAgentSession = await mcpAgent.CreateSessionAsync(CancellationToken.None);
Console.WriteLine("MCP Tool > Find number of open issues in microsoft/agent-framework repo");
ChatMessage mcpMessage = new ChatMessage(ChatRole.User, "Get the number of open issues in microsoft/agent-framework repo");
var mcpResponse = await mcpAgent.RunAsync(mcpMessage, mcpAgentSession);
Console.WriteLine(mcpResponse);


Installed Packages azure.ai.projects, 1.2.0-beta.5 Azure.Identity, 1.18.0-beta.2 Microsoft.Agents.AI, 1.0.0-rc1 Microsoft.Agents.AI.AzureAI, 1.0.0-rc1

Time Tool > The current date and time is March 3, 2026, 04:18:55 UTC. The timezone is Central Standard Time.



warning CS1702: Assuming assembly reference 'System.ClientModel, Version=1.8.0.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' used by 'Azure.Core' matches identity 'System.ClientModel, Version=1.8.1.0, Culture=neutral, PublicKeyToken=92742159e12e44c8' of 'System.ClientModel', you may need to supply runtime policy

